# Selection based on Wikipedia Events

In [1]:
import os
import re

import pke
import pickle
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
stoplist = stopwords.words('english')
ngram, top_n_keywords=3, 15
# please define your own elastic_curl_command (replace XXXnytcorpus_commandXXX if you use ElasticSearch)
#elastic_curl_command=r"""curl -u XXXnytcorpus_commandXXX?scroll=10m&size=25' ¥ -H 'Content-Type: application/json' ¥ --data-raw '{"query": {"simple_query_string":{"query":"query_keywords_string","fields" : ["body"]}}}'"""

In [2]:
def yake_keyword_extraction(sentence_text,stoplist,ngram=3,top_n_keywords=15):
    # 1. create a YAKE extractor.
    extractor = pke.unsupervised.YAKE()
    extractor.stoplist = stoplist
    # 2. load the content of the document.
    extractor.load_document(input=sentence_text,language='en', normalization=None)
    # 3. select {1-3}-grams not containing punctuation marks and not beginning/ending with a stopword as candidates.
    extractor.candidate_selection(n=ngram)
    # 4. weight the candidates using YAKE weighting scheme, a window (in words) for computing left/right contexts can be specified.
    window = 2
    use_stems = False
    extractor.candidate_weighting(window=window,use_stems=use_stems)
    # 5. get the 10-highest scored candidates as keyphrases.
    # redundant keyphrases are removed from the output using levenshtein distance and a threshold.
    threshold = 0.8
    keyphrases = extractor.get_n_best(n=top_n_keywords, threshold=threshold)
    keywords_list=[]
    for keyword_val_tuple in keyphrases:
        keywords_list.append(keyword_val_tuple[0])
    return keywords_list

def return_es_result_or(curl_command_string,key_w):
    add_key_string=""
    for k in key_w:
        add_key_string=add_key_string+r"\""+k+r"\"|"
    add_key_string=add_key_string[:-2]+"\""
    curl_command=curl_command_string.replace("XXX", add_key_string)
    es_result = os.popen(curl_command).readlines()
    doc_info_list=[]
    if len(es_result)==0:
        return doc_info_list
    for doc_text in es_result[0].split(r'_type":"_doc')[1:]:
        doc_info=re.findall(r'_id\":\"(\d+)\","_score\":(\d+\.\d+).*"body":".*","articleAbstract".*"publicationDate":"(\d+-\d+-\d+)"',doc_text)
        doc_info_list.extend(list(doc_info))
    return doc_info_list

In [7]:
wiki_event_list = pickle.load(open("data/Wiki_WebPage.pickle", "rb"))
print(len(wiki_event_list))
pd.DataFrame(wiki_event_list, columns=["No.", "Time", "Event-Text"]).head(5)

2733


,No.,Time,Event-Text
0,0,1987-01-02,Chadian–Libyan conflict – Battle of Fada: The ...
1,1,1987-01-03,Aretha Franklin becomes the first woman induct...
2,2,1987-01-04,1987 Maryland train collision: An Amtrak train...
3,3,1987-01-05,U.S. President Ronald Reagan undergoes prostat...
4,4,1987-01-08,The Dow Jones Industrial Average closes for th...


In [8]:
wiki_event_list

[['0',
  '1987-01-02',
  'Chadian–Libyan conflict – Battle of Fada: The Chadian army destroys a Libyan armoured brigade.'],
 ['1',
  '1987-01-03',
  'Aretha Franklin becomes the first woman inducted into the Rock and Roll Hall of Fame.'],
 ['2',
  '1987-01-04',
  '1987 Maryland train collision: An Amtrak train en route from Washington, D.C. to Boston collides with Conrail engines at Chase, Maryland, killing 16.'],
 ['3',
  '1987-01-05',
  'U.S. President Ronald Reagan undergoes prostate surgery, causing speculation about his physical fitness to continue in office.'],
 ['4',
  '1987-01-08',
  'The Dow Jones Industrial Average closes for the first time above 2,000, gaining 8.30 to close at 2,002.25.'],
 ['5',
  '1987-01-13',
  'New York mafiosi Anthony "Fat Tony" Salerno and Carmine Peruccia are sentenced to 100 years in prison for racketeering.'],
 ['6',
  '1987-01-15',
  'Hu Yaobang, General Secretary of the Communist Party of China, is forced into retirement by political conservatives

In [10]:
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm

for i, r in enumerate(tqdm(wiki_event_list, total=len(wiki_event_list))):
    event_text = r[2]
    keywords_list = yake_keyword_extraction(event_text, stoplist, ngram, top_n_keywords)
    doc_info = return_es_result_or(elastic_curl_command, keywords_list)    
    r.append(keywords_list)
    r.append(doc_info)
pd.DataFrame(wiki_event_list, columns=["No.","Time","Event-Text", "Keywords", "Doc-Info"]).head(5)
# The last column is the doc_info that obtained before
# Each element in doc_info is a triple, e.g. (853, 77.85112, 1987-01-04), representing (doc-id, BM25-score, doc-timestamp)
# The article text can then be easily obtained by using doc-id, which can then be used to generate questions

  0%|                                                                                                                                                                                                                                                          | 0/2733 [00:00<?, ?it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  0%|                                                                                                                                                                                                                                                  | 1/2733 [00:00<16:25,  2.77it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  0%|▏                                                                                                                                                                                        

  2%|███▌                                                                                                                                                                                                                                             | 41/2733 [00:12<13:33,  3.31it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  2%|███▋                                                                                                                                                                                                                                             | 42/2733 [00:13<13:31,  3.31it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  2%|███▊                                                        

  2%|█████▍                                                                                                                                                                                                                                           | 61/2733 [00:19<15:25,  2.89it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  2%|█████▍                                                                                                                                                                                                                                           | 62/2733 [00:19<15:23,  2.89it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  2%|█████▌                                                                                                                                                                                   

  4%|████████▊                                                                                                                                                                                                                                       | 101/2733 [00:31<13:02,  3.36it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  4%|████████▉                                                                                                                                                                                                                                       | 102/2733 [00:31<12:58,  3.38it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  4%|█████████                                                                                                                                                                                

  5%|████████████▍                                                                                                                                                                                                                                   | 141/2733 [00:44<14:34,  2.96it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  5%|████████████▍                                                                                                                                                                                                                                   | 142/2733 [00:44<13:51,  3.12it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
  5%|████████████▌                                                                                                                                                                            

...

 95%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏          | 2610/2733 [20:07<00:38,  3.24it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
 96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎          | 2611/2733 [20:07<00:37,  3.26it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
 96%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

 97%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋       | 2650/2733 [20:25<00:34,  2.44it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
 97%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊       | 2651/2733 [20:25<00:32,  2.55it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
 97%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

 98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏   | 2690/2733 [20:39<00:14,  3.03it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
 98%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎   | 2691/2733 [20:39<00:15,  2.70it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
 98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋| 2730/2733 [20:54<00:01,  2.56it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 2731/2733 [20:54<00:00,  2.68it/s]/bin/sh: -c: line 0: unexpected EOF while looking for matching `''
/bin/sh: -c: line 1: syntax error: unexpected end of file
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

ValueError: 5 columns passed, passed data had 7 columns

In [ ]:
#store the wiki_event_list
pickle.dump(wiki_event_list, open("data/wikievent_docinfo_list.pickle", "wb"))